# 02 - Prétraitement des jeux de données

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

## 1. Chargement des données

On recharge la table principale depuis zéro — ce notebook est indépendant du notebook EDA.

In [2]:
DATA_PATH = '../data/'

app_train = pd.read_csv(DATA_PATH + 'application_train.csv')

print(f'Dimensions : {app_train.shape}')

Dimensions : (307511, 122)


## 2. Nettoyage des anomalies

Certaines valeurs ne sont pas des NaN mais sont quand même incorrectes. On les corrige avant toute autre chose.

In [3]:
# DAYS_EMPLOYED : la valeur 365243 code les clients sans emploi
# On la remplace par NaN pour ne pas fausser les calculs
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)

# Vérification
print('Valeurs 365243 restantes :', (app_train['DAYS_EMPLOYED'] == 365243).sum())
print('NaN dans DAYS_EMPLOYED :', app_train['DAYS_EMPLOYED'].isna().sum())

Valeurs 365243 restantes : 0
NaN dans DAYS_EMPLOYED : 55374


**Analyse :** Les 55 374 valeurs aberrantes sont bien remplacées par NaN. Elles seront traitées à l'étape d'imputation comme n'importe quelle valeur manquante.

## 3. Encodage des variables catégorielles

Les modèles ML ne comprennent que des nombres. On doit convertir les colonnes texte en valeurs numériques.

- **Label Encoding** : pour les colonnes avec seulement 2 valeurs (ex: `Y/N`) — on les remplace par 0/1
- **One-Hot Encoding** : pour les colonnes avec plus de 2 valeurs (ex: `NAME_INCOME_TYPE`) — on crée une colonne binaire par modalité

In [4]:
# Séparation des colonnes catégorielles selon leur nombre de modalités
cols_cat = app_train.select_dtypes(include='object').columns.tolist()

le_cols  = [c for c in cols_cat if app_train[c].nunique() == 2]  # 2 modalités → Label Encoding
ohe_cols = [c for c in cols_cat if app_train[c].nunique() > 2]   # +2 modalités → One-Hot

print(f'Label Encoding ({len(le_cols)} colonnes) :', le_cols)
print(f'\nOne-Hot Encoding ({len(ohe_cols)} colonnes) :', ohe_cols)

Label Encoding (4 colonnes) : ['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE']

One-Hot Encoding (12 colonnes) : ['CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']


**Analyse :** 4 colonnes à Label Encoding (binaires : Yes/No, Cash/Revolving...) et 12 à One-Hot. 

In [5]:
# --- Label Encoding ---
le = LabelEncoder()
for col in le_cols:
    non_null = app_train[col].notna()
    # conversion en object d'abord pour accepter les entiers du LabelEncoder
    app_train[col] = app_train[col].astype(object)
    app_train.loc[non_null, col] = le.fit_transform(app_train.loc[non_null, col])

# --- One-Hot Encoding ---
app_train = pd.get_dummies(app_train, columns=ohe_cols, drop_first=True)

print(f'Dimensions après encodage : {app_train.shape}')

Dimensions après encodage : (307511, 230)


**Analyse :** On passe de 122 à 230 colonnes. L'augmentation vient du One-Hot Encoding qui a créé une colonne binaire par modalité (ex: `ORGANIZATION_TYPE` avait 57 modalités → 56 nouvelles colonnes). `drop_first=True` a supprimé une colonne par variable pour éviter la redondance.

## 4. Jointures avec les tables secondaires

Les tables secondaires ont **plusieurs lignes par client** (historique mensuel, multiples crédits...). On ne peut pas les joindre directement — on doit d'abord les **agréger** (1 ligne par client) puis faire un `left join` sur `SK_ID_CURR`.

### 4.1 Bureau — crédits dans d'autres banques

In [6]:
bureau = pd.read_csv(DATA_PATH + 'bureau.csv')

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_NB_CREDITS     = ('SK_ID_BUREAU', 'count'),         # nombre de crédits externes
    BUREAU_NB_ACTIFS      = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),  # crédits actifs
    BUREAU_DETTE_TOTALE   = ('AMT_CREDIT_SUM_DEBT', 'sum'),    # dette totale
    BUREAU_DETTE_MOYENNE  = ('AMT_CREDIT_SUM_DEBT', 'mean'),   # dette moyenne par crédit
    BUREAU_JOURS_MOYEN    = ('DAYS_CREDIT', 'mean'),           # ancienneté moyenne des crédits
).reset_index()

# Préfixe BUREAU_ déjà dans les noms → pas besoin de suffixe à la jointure
app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
# Les clients sans historique bureau auront des NaN → on impute par 0 (pas de crédit = 0)
app_train[bureau_agg.columns[1:]] = app_train[bureau_agg.columns[1:]].fillna(0)

print(f'Dimensions après bureau : {app_train.shape}')

Dimensions après bureau : (307511, 235)


**Analyse :** +5 colonnes issues de bureau. Le nombre de lignes reste 307 511 — le left join a bien conservé tous les clients.

### 4.2 Previous application — demandes passées chez Home Credit

In [7]:
prev = pd.read_csv(DATA_PATH + 'previous_application.csv')

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_NB_DEMANDES    = ('SK_ID_PREV', 'count'),
    PREV_NB_APPROUVEES  = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_NB_REFUSEES    = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_MONTANT_MOYEN  = ('AMT_APPLICATION', 'mean'),
    PREV_CREDIT_MOYEN   = ('AMT_CREDIT', 'mean'),
).reset_index()

prev_agg['PREV_TAUX_APPROBATION'] = prev_agg['PREV_NB_APPROUVEES'] / prev_agg['PREV_NB_DEMANDES']

app_train = app_train.merge(prev_agg, on='SK_ID_CURR', how='left')
app_train[prev_agg.columns[1:]] = app_train[prev_agg.columns[1:]].fillna(0)

print(f'Dimensions après previous_application : {app_train.shape}')

Dimensions après previous_application : (307511, 241)


**Analyse :** +6 colonnes (5 agrégats + le taux d'approbation calculé). On enchaîne avec les 3 tables restantes.

### 4.3 POS Cash, Installments, Credit Card

In [8]:
# --- POS CASH BALANCE ---
pos = pd.read_csv(DATA_PATH + 'POS_CASH_balance.csv')
pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_NB_MOIS      = ('MONTHS_BALANCE', 'count'),
    POS_RETARD_MOYEN = ('SK_DPD', 'mean'),
    POS_RETARD_MAX   = ('SK_DPD', 'max'),
).reset_index()

# --- INSTALLMENTS PAYMENTS ---
install = pd.read_csv(DATA_PATH + 'installments_payments.csv')
install['RETARD']  = install['DAYS_INSTALMENT'] - install['DAYS_ENTRY_PAYMENT']  # jours de retard
install['MANQUE']  = install['AMT_INSTALMENT'] - install['AMT_PAYMENT']          # montant non payé
install_agg = install.groupby('SK_ID_CURR').agg(
    INSTALL_NB_PAIEMENTS = ('AMT_INSTALMENT', 'count'),
    INSTALL_RETARD_MOYEN = ('RETARD', 'mean'),
    INSTALL_RETARD_MAX   = ('RETARD', 'max'),
    INSTALL_MANQUE_MOYEN = ('MANQUE', 'mean'),
).reset_index()

# --- CREDIT CARD BALANCE ---
cc = pd.read_csv(DATA_PATH + 'credit_card_balance.csv')
cc_agg = cc.groupby('SK_ID_CURR').agg(
    CC_NB_MOIS       = ('MONTHS_BALANCE', 'count'),
    CC_RETARD_MOYEN  = ('SK_DPD', 'mean'),
    CC_RETARD_MAX    = ('SK_DPD', 'max'),
    CC_SOLDE_MOYEN   = ('AMT_BALANCE', 'mean'),
).reset_index()

# --- JOINTURES ---
for agg_df in [pos_agg, install_agg, cc_agg]:
    app_train = app_train.merge(agg_df, on='SK_ID_CURR', how='left')
    app_train[agg_df.columns[1:]] = app_train[agg_df.columns[1:]].fillna(0)

print(f'Dimensions finales : {app_train.shape}')

Dimensions finales : (307511, 252)


**Analyse :** +11 colonnes issues des 3 tables (3 + 4 + 4). Dataset final : 307 511 clients × 252 features. Toutes les tables sont intégrées, aucune ligne perdue.

## 5. Traitement des valeurs manquantes

Maintenant que toutes les tables sont jointes, on traite les NaN sur le dataset complet. Première étape : visualiser les colonnes les plus incomplètes.

In [9]:
pct_nan = (app_train.isnull().mean() * 100).round(2)
pct_nan = pct_nan[pct_nan > 0].sort_values(ascending=False)

print(f'Colonnes avec NaN : {len(pct_nan)} / {app_train.shape[1]}\n')
print(pct_nan.to_string())

Colonnes avec NaN : 60 / 252

COMMONAREA_AVG                  69.87
COMMONAREA_MODE                 69.87
COMMONAREA_MEDI                 69.87
NONLIVINGAPARTMENTS_AVG         69.43
NONLIVINGAPARTMENTS_MODE        69.43
NONLIVINGAPARTMENTS_MEDI        69.43
LIVINGAPARTMENTS_MODE           68.35
LIVINGAPARTMENTS_MEDI           68.35
LIVINGAPARTMENTS_AVG            68.35
FLOORSMIN_AVG                   67.85
FLOORSMIN_MODE                  67.85
FLOORSMIN_MEDI                  67.85
YEARS_BUILD_MEDI                66.50
YEARS_BUILD_AVG                 66.50
YEARS_BUILD_MODE                66.50
OWN_CAR_AGE                     65.99
LANDAREA_AVG                    59.38
LANDAREA_MEDI                   59.38
LANDAREA_MODE                   59.38
BASEMENTAREA_MEDI               58.52
BASEMENTAREA_AVG                58.52
BASEMENTAREA_MODE               58.52
EXT_SOURCE_1                    56.38
NONLIVINGAREA_AVG               55.18
NONLIVINGAREA_MODE              55.18
NONLIVINGAREA_MEDI  

**Analyse :** 45 colonnes dépassent 40% de NaN — ce sont essentiellement des caractéristiques du bâtiment (surface, étages, caves...). Elles peuvent être supprimées sans perte métier majeure.

**`EXT_SOURCE_1` (56% NaN)** — c'était la 3ème variable la plus corrélée avec TARGET dans notre EDA. Supprimer ou imputer ? C'est un choix à justifier.

In [10]:
SEUIL = 0.4

# Colonnes à supprimer : >40% NaN, sauf EXT_SOURCE_1 qu'on garde (fort pouvoir prédictif)
cols_a_supprimer = [col for col in pct_nan[pct_nan > SEUIL * 100].index
                    if col != 'EXT_SOURCE_1']

app_train = app_train.drop(columns=cols_a_supprimer)

print(f'Colonnes supprimées : {len(cols_a_supprimer)}')
print(f'Dimensions : {app_train.shape}')
print(f'NaN restants : {app_train.isnull().sum().sum()}')

Colonnes supprimées : 45
Dimensions : (307511, 207)
NaN restants : 543868


**Analyse :** 45 colonnes supprimées, il reste 207 features. Les 543 868 NaN restants concernent des colonnes à moins de 40% — on va les imputer.

In [11]:
# Imputation par la médiane pour toutes les colonnes numériques avec NaN
for col in app_train.select_dtypes(include=[np.number]).columns:
    if app_train[col].isna().sum() > 0:
        app_train[col] = app_train[col].fillna(app_train[col].median())

print('NaN restants :', app_train.isnull().sum().sum())

NaN restants : 0


**Analyse :** Plus aucun NaN. Dataset propre et complet, prêt pour l'entraînement.

## 6. Feature Engineering

On crée de nouvelles variables à partir des colonnes existantes pour enrichir le modèle. L'idée : les **ratios** capturent mieux la situation financière relative d'un client que les valeurs absolues seules (un crédit de 100k€ n'a pas le même sens pour quelqu'un qui gagne 20k€ ou 80k€).

On supprime ensuite `DAYS_BIRTH` devenu redondant avec `AGE`.

In [12]:
# Ratio annuité / revenu : part du revenu consacrée au remboursement
app_train['RATIO_ANNUITE_REVENU'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']

# Ratio crédit / revenu : combien d'années de salaire pour rembourser
app_train['RATIO_CREDIT_REVENU'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']

# Ratio crédit / valeur du bien financé
app_train['RATIO_CREDIT_BIEN'] = app_train['AMT_CREDIT'] / app_train['AMT_GOODS_PRICE']

# Âge en années (DAYS_BIRTH est négatif)
app_train['AGE'] = -app_train['DAYS_BIRTH'] / 365
app_train = app_train.drop(columns=['DAYS_BIRTH'])

print(f'Dimensions après FE : {app_train.shape}')
print(app_train[['RATIO_ANNUITE_REVENU', 'RATIO_CREDIT_REVENU', 'RATIO_CREDIT_BIEN', 'AGE']].describe().round(2))

Dimensions après FE : (307511, 210)
       RATIO_ANNUITE_REVENU  RATIO_CREDIT_REVENU  RATIO_CREDIT_BIEN        AGE
count             307511.00            307511.00          307511.00  307511.00
mean                   0.18                 3.96               1.12      43.94
std                    0.09                 2.69               0.13      11.96
min                    0.00                 0.00               0.15      20.52
25%                    0.11                 2.02               1.00      34.01
50%                    0.16                 3.27               1.12      43.15
75%                    0.23                 5.16               1.20      53.92
max                    1.88                84.74               6.00      69.12


**Analyse :** 4 nouvelles features créées (+4 colonnes). `RATIO_CREDIT_REVENU` moyen à 3.96 — un client emprunte en moyenne ~4 ans de salaire. `RATIO_CREDIT_BIEN` > 1 en moyenne (1.12) : le crédit est légèrement supérieur au prix du bien, ce qui inclut les frais annexes.

## 7. Sauvegarde du dataset final

In [13]:
# Vérifications avant sauvegarde
print(f'Dimensions : {app_train.shape}')
print(f'NaN restants : {app_train.isna().sum().sum()}')
print(f'Types de colonnes :\n{app_train.dtypes.value_counts()}')
print(f'\nTaux de défaut : {app_train["TARGET"].mean()*100:.2f}%')

Dimensions : (307511, 210)
NaN restants : 0
Types de colonnes :
bool       120
float64     48
int64       39
object       3
Name: count, dtype: int64

Taux de défaut : 8.07%


In [14]:
# Identifier les colonnes object restantes
print(app_train.select_dtypes('object').columns.tolist())

['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']


In [15]:
for col in ['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']:
    app_train[col] = pd.to_numeric(app_train[col]).astype(int)

print(app_train[['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']].dtypes)

NAME_CONTRACT_TYPE    int64
FLAG_OWN_CAR          int64
FLAG_OWN_REALTY       int64
dtype: object


In [16]:
print(app_train.dtypes.value_counts())

bool       120
float64     48
int64       42
Name: count, dtype: int64


In [17]:
import os

os.makedirs('../data/processed', exist_ok=True)

app_train.to_csv('../data/processed/dataset_final.csv', index=False)

print(f'Dataset sauvegardé : {app_train.shape}')

Dataset sauvegardé : (307511, 210)


**Analyse :** Dataset final sauvegardé dans `data/processed/dataset_final.csv` — 307 511 clients × 210 features, 0 NaN, 100% numérique.

**Prochaine étape →** `03_training_mlflow.ipynb` : entraînement des modèles avec tracking MLflow.